# Layer 1, flat version

Same logic as `src/` (config.py, retrieval.py, nodes.py, state.py, graph.py), but written the way you write your own notebooks: one file, top-level variables, no imports between your own code.

This reuses the `.chroma` folder you already built by running `python main.py ingest` in the project root, so run that first if you haven't. This notebook only *reads* that index — it doesn't rebuild it.

At the bottom there's a cell-by-cell map back to the real `src/` files, and a suggested exercise.

In [ ]:
from pathlib import Path
from typing import TypedDict, List

from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
load_dotenv()

# In src/config.py these live in a separate file so every module can share
# them. Here they're just plain variables at the top of the notebook,
# same as how you'd set them up in any of your other notebooks.
LLM_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# The notebook lives in notebooks/, the .chroma folder lives one level up,
# in the project root — because that's where `python main.py ingest` put it.
CHROMA_DIR = Path("..") / ".chroma"
COLLECTION_NAME = "agentic_rag"

DENSE_K = 8    # candidates pulled from the vector store
SPARSE_K = 8   # candidates pulled from BM25
RRF_K = 60     # Reciprocal Rank Fusion constant (standard default)
TOP_K = 5      # final chunks handed to the generator

In [ ]:
# Plain top-level model objects, exactly like your other notebooks
# (e.g. `model = ChatOpenAI(model='gpt-4o-mini')` in conditional_workflow.ipynb).
# No lazy-loading wrapper function — that was unnecessary complexity I'd
# added in src/nodes.py, not something LangGraph needs.
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

In [ ]:
# Load the persisted vector store from disk. This is the key idea that's
# different from a normal notebook: `python main.py ingest` ran in a
# SEPARATE process, earlier, and wrote its results to the .chroma folder.
# We're not re-embedding anything here — we're reconnecting to what's
# already on disk, the same way you'd re-open a saved file.
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

raw = vector_store.get()  # -> {'ids', 'documents', 'metadatas', ...}
all_docs = [
    Document(page_content=text, metadata=meta or {})
    for text, meta in zip(raw["documents"], raw["metadatas"])
]

if not all_docs:
    raise SystemExit("Chroma is empty. Run `python main.py ingest` in the project root first.")

len(all_docs)

In [ ]:
# BM25 is a keyword-based (sparse) retriever, built in-memory from the same
# documents already sitting in Chroma. There's no second index saved to
# disk — it's rebuilt fresh every time this notebook (or the app) starts.
bm25 = BM25Retriever.from_documents(all_docs)
bm25.k = SPARSE_K

In [ ]:
def doc_key(doc: Document) -> str:
    """A stable identity for de-duplicating the same chunk across both retrievers."""
    src = doc.metadata.get("source", "")
    idx = doc.metadata.get("start_index", "")
    return f"{src}::{idx}::{hash(doc.page_content)}"


def reciprocal_rank_fusion(ranked_lists, k: int = RRF_K):
    """Fuse multiple ranked lists. RRF score = sum over lists of 1/(k + rank)."""
    scores = {}
    registry = {}
    for ranked in ranked_lists:
        for rank, doc in enumerate(ranked):
            key = doc_key(doc)
            registry[key] = doc
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank + 1)
    ordered = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    return [registry[key] for key, _ in ordered]

In [ ]:
def hybrid_retrieve(question: str, top_k: int = TOP_K):
    """Return the top_k chunks after dense + sparse retrieval and RRF fusion."""
    dense = vector_store.similarity_search(question, k=DENSE_K)
    sparse = bm25.invoke(question)
    fused = reciprocal_rank_fusion([dense, sparse])
    return fused[:top_k]

In [ ]:
# Quick sanity check before wiring up the graph — swap in a question that
# actually matches your corpus.
test_docs = hybrid_retrieve("What is the transformer architecture?")
for d in test_docs:
    print(d.metadata.get("source"), "-", d.page_content[:80].replace("\n", " "))

In [ ]:
# Same idea as CricketState / ReviewState in your other notebooks:
# a TypedDict describing what flows through the graph.
class GraphState(TypedDict, total=False):
    question: str
    documents: List[Document]
    generation: str

In [ ]:
GENERATE_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a precise question-answering assistant. Answer ONLY from the "
            "numbered context below. After each claim, cite the supporting source(s) "
            "inline like [1] or [1][3]. If the context does not contain the answer, "
            "reply exactly: \"I don't have enough information in the provided "
            "documents to answer that.\" Do not use outside knowledge.\n\n"
            "Context:\n{context}",
        ),
        ("human", "{question}"),
    ]
)


def format_context(documents) -> str:
    """Number each chunk so the model can cite [1], [2], ... and we can map back."""
    blocks = []
    for i, d in enumerate(documents, start=1):
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page")
        loc = src + (f", p.{page}" if page is not None else "")
        blocks.append(f"[{i}] ({loc})\n{d.page_content}")
    return "\n\n".join(blocks)

In [ ]:
# Node functions — same shape as find_sentiment/positive_response in
# conditional_workflow.ipynb: take state in, return a dict of updates.
# They reach straight for the top-level `llm`, `vector_store`, `bm25`
# variables defined above. No `self`, no imported module, no lazy singleton.

def retrieve(state: GraphState) -> GraphState:
    docs = hybrid_retrieve(state["question"])
    return {"documents": docs}


def generate(state: GraphState) -> GraphState:
    context = format_context(state["documents"])
    chain = GENERATE_PROMPT | llm
    answer = chain.invoke({"context": context, "question": state["question"]}).content
    return {"generation": answer}

In [ ]:
# Same graph-building pattern as cricket.ipynb / conditional_workflow.ipynb:
# create StateGraph, add_node, add_edge, compile.
graph = StateGraph(GraphState)

graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

workflow = graph.compile()

In [ ]:
graph.compile()

In [ ]:
initial_state = {
    "question": "What is the transformer architecture?"
}

result = workflow.invoke(initial_state)

print("=== ANSWER ===\n")
print(result["generation"])

print("\n=== SOURCES ===\n")
for i, d in enumerate(result["documents"], start=1):
    src = d.metadata.get("source", "unknown")
    page = d.metadata.get("page")
    loc = src + (f", p.{page}" if page is not None else "")
    print(f"[{i}] {loc}")

## Cell-by-cell map back to `src/`

| This notebook | Same logic in the real project |
|---|---|
| `cell-env-config` (LLM_MODEL, CHROMA_DIR, ...) | `src/config.py` |
| `cell-load-chroma`, `cell-bm25` | `src/retrieval.py` — `_load_chroma`, `_ensure_loaded` |
| `cell-rrf`, `cell-hybrid-retrieve` | `src/retrieval.py` — `reciprocal_rank_fusion`, `hybrid_retrieve` |
| `cell-state` | `src/state.py` — `GraphState` |
| `cell-prompt`, `cell-nodes` | `src/nodes.py` — `GENERATE_PROMPT`, `retrieve`, `generate` |
| `cell-graph` | `src/graph.py` — `build_graph` |
| `cell-invoke` | `main.py` — `run_query` |

The logic is identical, line for line. The only real differences: the real project splits it across files and reloads `vector_store`/`bm25` fresh on every run of `main.py` (a new process each time), while a notebook keeps everything alive in one kernel until you restart it.

**Exercise, when you're ready:** try turning this single notebook back into `config.py`, `retrieval.py`, `state.py`, `nodes.py`, `graph.py`, and `main.py` yourself — cutting each section into its own file and wiring up the imports between them (`from . import config`, `from .state import GraphState`, `from src import ingestion`, etc.). Don't peek at the real `src/` files while you do it. Then compare your version against mine and we'll go through any differences together.

That splitting exercise — not this notebook — is where the actual packaging skill gets built. This notebook was just to prove the *logic* was never the problem.